# 낯씨 OCR 검출(det) 모델 파인튜닝 — Colab GPU

1. 런타임 → 런타임 유형 변경 → **GPU (T4)**
2. 로컬에서 `pack_for_colab.py`로 만든 `ocr-train.colab.zip`을 구글 드라이브 `MyDrive/natssi/`에 올린다
3. 위에서부터 차례로 실행

학습 결과(가중치·로그)는 **드라이브**(`MyDrive/natssi/output/`)에 바로 저장한다 → Colab 연결이 끊겨도 남고, 다시 실행하면 이어서 학습한다.
이 노트북에는 사용자 사진이 들어가지 않는다(팀이 준비한 학습 데이터만).

In [ ]:
# ── 설정 ──
ZIP = '/content/drive/MyDrive/natssi/ocr-train.colab.zip'
OUT = '/content/drive/MyDrive/natssi/output/natssi_det'   # 가중치·로그 저장(드라이브)
EPOCHS = None        # None이면 yml 값 그대로. GPU라 늘리고 싶으면 숫자로(예: 20)


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
import json, os, zipfile
os.makedirs(OUT, exist_ok=True)
meta = json.loads(zipfile.ZipFile(ZIP).read('colab_meta.json'))
print(meta)

In [ ]:
# PaddleOCR 같은 커밋으로 받고, 그 위에 학습 데이터를 같은 폴더 구조로 푼다
%cd /content
!rm -rf ocr-train && mkdir ocr-train
!git clone -q https://github.com/PaddlePaddle/PaddleOCR.git ocr-train/PaddleOCR
if meta['paddleocr_commit']:
    !cd ocr-train/PaddleOCR && git fetch -q --depth 1 origin {meta['paddleocr_commit']} 2>/dev/null; git checkout -q {meta['paddleocr_commit']} && git log -1 --oneline
!cd ocr-train && unzip -q -o {ZIP} && ls

In [ ]:
# PaddlePaddle GPU 판 설치(로컬과 같은 버전을 먼저 시도). 설치 뒤 '런타임 다시 시작'이 뜨면 눌러 주고 이 칸부터 다시.
ver = meta['paddle']
idx = 'https://www.paddlepaddle.org.cn/packages/stable/cu126/'
r = os.system(f'pip install -q paddlepaddle-gpu=={ver} -i {idx}') if ver else 1
if r != 0:
    print('같은 버전을 못 받아 최신 GPU 판으로 설치합니다')
    os.system(f'pip install -q paddlepaddle-gpu -i {idx}')
!cd /content/ocr-train/PaddleOCR && pip install -q -r requirements.txt
import paddle; paddle.utils.run_check(); print(paddle.__version__, 'GPU' if paddle.device.is_compiled_with_cuda() else 'CPU!')

In [ ]:
# yml 정리: 평가를 에폭마다, 저장은 드라이브로, 이어서 학습
import math, yaml
os.chdir('/content/ocr-train/PaddleOCR')
yml = '../' + meta['yml']
c = yaml.safe_load(open(yml, encoding='utf-8'))
if EPOCHS:
    c['Global']['epoch_num'] = EPOCHS
    for tr in c['Train']['dataset']['transforms']:
        for k in ('MakeBorderMap', 'MakeShrinkMap'):
            if isinstance(tr, dict) and k in tr and tr[k] and 'total_epoch' in tr[k]:
                tr[k]['total_epoch'] = EPOCHS
n = sum(1 for f in c['Train']['dataset']['label_file_list'] for l in open(f, encoding='utf-8') if l.strip())
bs = c['Train']['loader']['batch_size_per_card']
steps = max(1, math.ceil(n / bs))
c['Global'].update(use_gpu=True, save_model_dir=OUT + '/', eval_batch_step=[0, steps])
c['Global']['checkpoints'] = OUT + '/latest' if os.path.exists(OUT + '/latest.pdparams') else None
for d in ('Train', 'Eval'):                       # Colab CPU는 2개뿐
    c[d]['loader']['num_workers'] = min(c[d]['loader'].get('num_workers', 2), 2)
yaml.safe_dump(c, open('../colab.yml', 'w'), allow_unicode=True, sort_keys=False)
print(f"학습 {n}장, 배치 {bs}, 에폭당 {steps}스텝, 에폭 {c['Global']['epoch_num']}, 이어서: {bool(c['Global']['checkpoints'])}")

In [ ]:
!cd /content/ocr-train/PaddleOCR && python tools/train.py -c ../colab.yml 2>&1 | tee -a {OUT}/train.log | grep -E --line-buffered "epoch: \[|best metric|hmean|save model|Error|error" 

In [ ]:
!grep -E "best metric|hmean" {OUT}/train.log | tail -5
!ls -la {OUT}

## 추론 모델 → ONNX (RapidOCR/Space에서 쓰는 형식)

In [ ]:
%cd /content/ocr-train/PaddleOCR
best = OUT + '/best_accuracy' if os.path.exists(OUT + '/best_accuracy.pdparams') else OUT + '/latest'
INF = OUT + '/inference'
!python tools/export_model.py -c ../colab.yml -o Global.pretrained_model={best} Global.save_inference_dir={INF}
!pip install -q paddle2onnx onnxruntime
mf = 'inference.json' if os.path.exists(INF + '/inference.json') else 'inference.pdmodel'
!paddle2onnx --model_dir {INF} --model_filename {mf} --params_filename inference.pdiparams --save_file {OUT}/natssi_det.onnx --opset_version 14
import onnxruntime as ort
s = ort.InferenceSession(f'{OUT}/natssi_det.onnx'); print('ONNX OK', [i.shape for i in s.get_inputs()])
print('드라이브:', OUT + '/natssi_det.onnx')